In [ ]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

from sklearn.metrics.pairwise import cosine_similarity

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
ARTIFACTS_DIR = Path.cwd().parent / "artifacts"
QDRANT_DIR = Path.cwd().parent / "artifacts" / "qdrant"

In [ ]:
embeddings = np.load(ARTIFACTS_DIR / "embeddings.npy")
metadata = pd.read_parquet(ARTIFACTS_DIR / "metadata.parquet")
config = json.load(open(ARTIFACTS_DIR / "embedding_config.json", "r"))

print(f"Embeddings shape: {embeddings.shape}")
print(f"Metadata shape: {metadata.shape}")
print(f"Config: {config}")

In [ ]:
COLLECTION_NAME = "mirx_qdrant_integration"
TOP_K = 5
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
EMBEDDING_MODEL = config["embedding_model"]
BATCH_SIZE = 1000

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Embedding dimension: {embedding_model.get_embedding_dimension()}")

In [ ]:
client = QdrantClient(path = QDRANT_DIR)

existing_collections = [collection.name for collection in client.get_collections().collections]

if COLLECTION_NAME not in existing_collections:
    client.create_collection(
        collection_name = COLLECTION_NAME,
        vectors_config = VectorParams(
            size = embedding_model.get_embedding_dimension(),
            distance = Distance.COSINE
        )
    )

    print(
        f"Created collection: {COLLECTION_NAME}"
    )

else:

    print(
        f"Collection already exists: {COLLECTION_NAME}"
    )

In [ ]:
collection_info = client.get_collection(collection_name = COLLECTION_NAME)

existing_points = collection_info.points_count

print(f"Existing points in collection: {existing_points}")
print(f"Expected points to add: {len(embeddings)}")

In [ ]:
if existing_points < len(embeddings):

    ingestion_start = time.perf_counter()

    for start in tqdm(range(0, len(embeddings), BATCH_SIZE), desc="Uploading vectors"):

        end = min(start + BATCH_SIZE, len(embeddings))

        points = []

        for idx in range(start, end):

            payload = metadata.iloc[idx].to_dict()

            # Convert NumPy types to Python-compatible values
            payload = {
                key: (
                    value.item()
                    if isinstance(value, np.generic)
                    else value
                )
                for key, value in payload.items()
            }

            points.append(
                PointStruct(
                    id=idx,
                    vector=embeddings[idx].tolist(),
                    payload=payload
                )
            )

        client.upsert(collection_name=COLLECTION_NAME, points=points)

    ingestion_time = (time.perf_counter() - ingestion_start)

    print(f"\nIngestion completed in "f"{ingestion_time:.2f} seconds.")

else:

    print(
        "Qdrant already contains the expected "
        "number of vectors."
    )

In [ ]:
collection_info = client.get_collection(collection_name=COLLECTION_NAME)

print("=" * 60)
print("QDRANT COLLECTION")
print("=" * 60)

print("Collection:", COLLECTION_NAME)
print("Points:", collection_info.points_count)
print("Vectors:", len(embeddings))
print("Dimension:", config["embedding_dimension"])

assert (
    collection_info.points_count
    == len(embeddings)
), "Qdrant point count mismatch!"

In [ ]:
def retrieve_v2(query, top_k = TOP_K):
    query_embedding = embedding_model.encode(query, convert_to_numpy = True, normalize_embeddings = True)

    search_result = client.query_points(COLLECTION_NAME, query = query_embedding.tolist(), limit = top_k)

    return search_result.points

In [ ]:
query = "What are the alternatives to paracetamol?"

results = retrieve_v2(query, top_k = 5)

for i, result in enumerate(results):

    print("=" * 80)
    print(f"RESULT #{i + 1}")
    print("Score:", result.score)
    print()

    print(result.payload["document"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

llm_model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype = "auto", device_map = "auto")

llm_model.eval()

In [ ]:
SYSTEM_PROMPT = """
You are MIRx, a medical information retrieval assistant.

Your task is to answer questions using ONLY the information
provided in the retrieved context.

Rules:

1. Do not invent information.
2. Do not introduce medical facts that are absent from the context.
3. If the retrieved context is insufficient, say so clearly.
4. Keep the answer concise and well structured.
5. When possible, mention the relevant drug or medical entity.
6. Do not make diagnoses or provide personalized medical treatment.
7. Clearly distinguish information found in the retrieved context
   from uncertainty.
"""

In [ ]:
def build_context(results):

    context_parts = []

    for i, result in enumerate(results):

        document = result.payload.get(
            "document",
            ""
        )

        context_parts.append(
            f"""
SOURCE {i + 1}
Similarity: {result.score:.4f}

{document}
"""
        )

    return "\n".join(context_parts)

In [ ]:
def generate_with_qwen(query, context):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
Retrieved medical information:

{context}

User question:

{query}

Using only the retrieved medical information above,
answer the user's question.
"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    )

    inputs = inputs.to(next(llm_model.parameters()).device)

    input_length = inputs["input_ids"].shape[-1]

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens = 300,
            do_sample = False
        )

    generated_tokens = outputs[
        0,
        input_length:
    ]

    response = tokenizer.decode(generated_tokens, skip_special_tokens = True)

    return response.strip()

In [ ]:
def mirx_v2(query, top_k = TOP_K):

    total_start = time.perf_counter()

    # -----------------------------------------
    # Query embedding + retrieval
    # -----------------------------------------

    retrieval_start = time.perf_counter()

    results = retrieve_v2(query, top_k = top_k)

    retrieval_time = (time.perf_counter() - retrieval_start)

    # -----------------------------------------
    # Context construction
    # -----------------------------------------

    context = build_context(results)

    # -----------------------------------------
    # Qwen generation
    # -----------------------------------------

    generation_start = time.perf_counter()

    answer = generate_with_qwen(query, context)

    generation_time = (time.perf_counter() - generation_start)

    total_time = (time.perf_counter()- total_start)

    return {
        "answer": answer,
        "results": results,
        "context": context,
        "retrieval_time": retrieval_time,
        "generation_time": generation_time,
        "total_time": total_time
    }

In [ ]:
query = "What are the alternatives to paracetamol?"

result = mirx_v2(query, top_k = TOP_K)

print("=" * 80)
print("MIRx RESPONSE")
print("=" * 80)

print(result["answer"])

print("\n" + "=" * 80)
print("LATENCY")
print("=" * 80)

print(f"Retrieval: "f"{result['retrieval_time'] * 1000:.2f} ms")

print(f"Generation: "f"{result['generation_time']:.2f} s")

print(f"Total: "f"{result['total_time']:.2f} s")

In [ ]:
print("=" * 80)
print("RETRIEVED SOURCES")
print("=" * 80)

for i, result in enumerate(result["results"]):

    print(f"\n[{i + 1}] " f"Similarity: {result.score:.4f}")

    print(result.payload["document"])

In [ ]:
# ============================================================
# Multiple-query evaluation
# ============================================================

test_queries = [
    "What are the alternatives to paracetamol?",
    "What are the side effects of aspirin?",
    "Which medicines are used for hypertension?",
    "What are the substitutes for ibuprofen?",
    "What drugs belong to the antibiotic class?"
]

evaluation_results = []

for query in test_queries:

    print("=" * 80)
    print("QUERY:", query)

    result = mirx_v2(
        query,
        top_k=5
    )

    print("\nANSWER:")
    print(result["answer"])

    evaluation_results.append({
        "query": query,
        "retrieval_time_ms":
            result["retrieval_time"] * 1000,
        "generation_time_s":
            result["generation_time"],
        "total_time_s":
            result["total_time"]
    })

In [ ]:
# ============================================================
# V2 benchmark summary
# ============================================================

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)

print("\nAverage retrieval latency:", evaluation_df["retrieval_time_ms"].mean())

print("Average generation latency:", evaluation_df["generation_time_s"].mean())

print("Average total latency:", evaluation_df["total_time_s"].mean())